# Drive Failure Trend — Gold View

## Purpose
Provide a daily view of Backblaze drive observations and reported failures across source periods.

**Source:** `backblaze_data.sot_silver.drive_daily_stats`   
**Target:** `backblaze_data.sot_gold_views.drive_failure_trend_vw`  
**Grain:** One row per date and source period.  
**Calculation:**
Daily observed failure percentage =
(Failed drive observations / Total drive observations) * 100  

**Layer Purpose:** Gold — Business-facing analytical view.

## Business Metrics

- Total drive observations per date and source period.
- Number of observations where failure = 1.
- Daily observed failure percentage.

In [0]:
%sql
-- Purpose: Aggregate the Silver records into a business-facing daily failure trend.
CREATE OR REPLACE VIEW backblaze_data.sot_gold_views.drive_failure_trend_vw AS
SELECT
    date,
    _period,

    COUNT(*) AS total_drive_observations,

    SUM(
        CASE
            WHEN failure = 1 THEN 1
            ELSE 0
        END
    ) AS failed_drive_observations,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN failure = 1 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS daily_failure_rate_pct

FROM backblaze_data.sot_silver.drive_daily_stats
GROUP BY
    date,
    _period;

In [0]:
%sql
ALTER VIEW backblaze_data.sot_gold_views.drive_failure_trend_vw
SET TAGS (
    'layer' = 'gold',
    'domain' = 'drive_reliability',
    'subject' = 'failure_analysis',
    'refresh_type' = 'on_query'
);

In [0]:
%sql
COMMENT ON VIEW backblaze_data.sot_gold_views.drive_failure_trend_vw IS
'Provides daily Backblaze drive observation counts, failure counts, and observed failure percentages by observation date and source period. This metric is a daily observed failure   percentage and is not an annualized failure rate.';

In [0]:
%sql
COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_failure_trend_vw.date
IS 'Calendar date on which the drive observation was recorded.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_failure_trend_vw._period
IS 'Source dataset period associated with the observation, such as 2016_Q1 or 2018_Q1.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_failure_trend_vw.total_drive_observations
IS 'Total number of drive observations recorded for the date and source period.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_failure_trend_vw.failed_drive_observations
IS 'Number of drive observations where the failure indicator equals 1.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_failure_trend_vw.daily_failure_rate_pct
IS 'Percentage of observed drive records with failure equal to 1 for the date and source period. Calculated as failed observations divided by total observations multiplied by 100. Not an annualized failure rate.';

In [0]:
%sql
SELECT *
    FROM backblaze_data.sot_gold_views.drive_failure_trend_vw
    ORDER BY date, _period
LIMIT 20;

In [0]:
%sql
DESCRIBE TABLE EXTENDED
backblaze_data.sot_gold_views.drive_failure_trend_vw;